# 01. Bronze Ingestion & Telemetry Profiling (42.4M Behavioral Events)

### Architectural Purpose
In this notebook, we establish our **Tier 1 (Raw Event Logs)** baseline using the public **REES46 Multi-Category Behavioral Event Dataset** (`2019-Oct.csv.gz`, ~42.45 million stateless events). 

**Why this matters for LLM & Agentic Benchmarking:**
* **No Silent Schema Inference:** Letting Spark infer schemas over multi-million-row raw logs wastes compute and risks misclassifying high-cardinality IDs or timestamps. We enforce an explicit `StructType` schema at read time.
* **Delta Lake Storage Optimization:** Converting raw gzipped CSV logs (~1.4 GB compressed / ~5.67 GB uncompressed) into an uncompressed **Delta table** (`workspace.default.bronze_events`) enables Parquet columnar compression (~1.2 GB on disk) and transaction-log min/max data skipping without introducing small-file partitioning overhead.
* **Baseline Anomaly Audit:** Before building our Tier 2 (Semantic Layer) and Tier 3 (Ontology) abstractions, we profile null rates and funnel distributions to document the exact data-quality traps that cause naive Text-to-SQL agents to hallucinate.  


In [1]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
from pyspark.sql import functions as F

raw_path = "/Volumes/workspace/default/raw_events/2019-Oct.csv.gz"

event_schema = StructType([
    StructField("event_time", StringType(), True), 
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

df_raw = (
    spark.read
    .option("header", "true")
    .schema(event_schema)
    .csv(raw_path)
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .drop("event_time")
)

display(df_raw.limit(100))


event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_ts
view,44600062,2103807459595387724,None,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c,2019-10-01 00:00:00
view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc,2019-10-01 00:00:00
view,17200506,2053013559792632471,furniture.living_room.sofa,None,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8,2019-10-01 00:00:01
view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,2019-10-01 00:00:01
view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,2019-10-01 00:00:04
view,1480613,2053013561092866779,computers.desktop,pulser,908.62,512742880,0d0d91c2-c9c2-4e81-90a5-86594dec0db9,2019-10-01 00:00:05
view,17300353,2053013553853497655,None,creed,380.96,555447699,4fe811e9-91de-46da-90c3-bbd87ed3a65d,2019-10-01 00:00:08
view,31500053,2053013558031024687,None,luminarc,41.16,550978835,6280d577-25c8-4147-99a7-abc6048498d6,2019-10-01 00:00:08
view,28719074,2053013565480109009,apparel.shoes.keds,baden,102.71,520571932,ac1cd4e5-a3ce-4224-a2d7-ff660a105880,2019-10-01 00:00:10
view,1004545,2053013555631882655,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb,2019-10-01 00:00:11


In [2]:
(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_events")
)

print("Saved bronze_events successfully!")


Saved bronze_events successfully!


In [3]:
bronze = spark.table("workspace.default.bronze_events")

print(f"Total raw events: {bronze.count():,}")

# Breakdown by event_type
display(
    bronze.groupBy("event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("user_id").alias("distinct_users"),
        F.countDistinct("user_session").alias("distinct_sessions")
    )
    .orderBy(F.desc("event_count"))
)

# Check null rates (especially category_code, brand, and user_session)
null_summary = bronze.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100, 2).alias(f"{c}_null_pct")
    for c in bronze.columns
])
display(null_summary)


Total raw events: 42,448,764


event_type,event_count,distinct_users,distinct_sessions
view,40779399,3022130,9242653
cart,926516,337117,573097
purchase,742849,347118,629560


event_type_null_pct,product_id_null_pct,category_id_null_pct,category_code_null_pct,brand_null_pct,price_null_pct,user_id_null_pct,user_session_null_pct,event_ts_null_pct
0,0,0,31.84,14.4,0,0,0,0
